# Horizon IRC External Reads & Writes Demo
### Querying Snowflake-Managed Iceberg Tables with Apache Spark via Horizon Catalog

This notebook demonstrates:
- **External Reads** of Snowflake-managed Iceberg tables from Spark via the Horizon IRC REST API
- **RBAC enforcement** across two personas (DATA_ENGINEER vs DATA_ANALYST)
- **Masking policies** on PII columns (email, phone, SSN)
- **Row access policies** filtering data by region
- **External Writes** back to Iceberg tables from Spark
- **Two auth methods**: PAT (Programmatic Access Token) and Key-Pair Authentication

**Reference**: [Snowflake Documentation](https://docs.snowflake.com/en/user-guide/tables-iceberg-query-using-external-query-engine-snowflake-horizon)

## 1. Environment Setup
Install PySpark and Java inside the notebook container runtime.

In [ ]:
!pip install -q pyspark==3.5.0 findspark==2.0.1
!DEBIAN_FRONTEND=noninteractive apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openjdk-11-jdk-headless 2>&1 | tail -1

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

import findspark
findspark.init()
print(f"Spark home: {findspark.find()}")
print(f"JAVA_HOME:  {os.environ['JAVA_HOME']}")

## 2. Configuration
Update these values for your Snowflake account. Run `01_admin_setup.sql` first to create the environment.

In [ ]:
# ── ACCOUNT CONFIGURATION ──────────────────────────────────────────────────────
# Replace underscores (_) in account name with hyphens (-) for the URL.
ACCOUNT_IDENTIFIER = "<YOUR_ORG>-<YOUR_ACCOUNT>"
CATALOG_URI        = f"https://{ACCOUNT_IDENTIFIER}.snowflakecomputing.com/polaris/api/catalog"
DATABASE_NAME      = "ICEBERG_DEMO_DB"   # must be UPPER CASE
AWS_REGION         = "us-west-2"

# ── PAT TOKENS (from 01_admin_setup.sql step 6) ───────────────────────────────
ENGINEER_PAT = "<PASTE_ENGINEER_PAT_HERE>"
ANALYST_PAT  = "<PASTE_ANALYST_PAT_HERE>"

# ── ICEBERG / SPARK VERSIONS ──────────────────────────────────────────────────
ICEBERG_VERSION = "1.9.1"

print(f"Horizon IRC endpoint: {CATALOG_URI}")
print(f"Database:             {DATABASE_NAME}")
print(f"Region:               {AWS_REGION}")

## 3. Spark Session Helper
Reusable function to create a Spark session connected to Horizon IRC.

In [ ]:
from pyspark.sql import SparkSession

def create_horizon_spark_session(token, role, auth_type="pat"):
    """Create a Spark session connected to Snowflake Horizon IRC.

    Args:
        token: PAT token string or OAuth access token
        role: Snowflake role name (e.g. 'DATA_ENGINEER')
        auth_type: 'pat' uses .credential, 'token' uses .token (for key-pair/OAuth)
    """
    try:
        spark.stop()
    except:
        pass

    catalog = "horizon"
    token_config_key = "credential" if auth_type == "pat" else "token"

    session = (
        SparkSession.builder
          .appName(f"HorizonIRC-{role}")
          .master("local[*]")
          .config("spark.ui.port", "0")
          .config("spark.driver.bindAddress", "127.0.0.1")
          .config("spark.driver.host", "127.0.0.1")
          .config("spark.driver.port", "0")
          .config("spark.blockManager.port", "0")
          .config("spark.ui.showConsoleProgress", "false")
          .config(
              "spark.jars.packages",
              f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{ICEBERG_VERSION},"
              f"org.apache.iceberg:iceberg-aws-bundle:{ICEBERG_VERSION}"
          )
          .config("spark.sql.extensions",
                  "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
          .config("spark.sql.defaultCatalog", catalog)
          .config(f"spark.sql.catalog.{catalog}",
                  "org.apache.iceberg.spark.SparkCatalog")
          .config(f"spark.sql.catalog.{catalog}.type", "rest")
          .config(f"spark.sql.catalog.{catalog}.uri", CATALOG_URI)
          .config(f"spark.sql.catalog.{catalog}.warehouse", DATABASE_NAME)
          .config(f"spark.sql.catalog.{catalog}.{token_config_key}", token)
          .config(f"spark.sql.catalog.{catalog}.scope",
                  f"session:role:{role}")
          .config(f"spark.sql.catalog.{catalog}.header.X-Iceberg-Access-Delegation",
                  "vended-credentials")
          .config(f"spark.sql.catalog.{catalog}.client.region", AWS_REGION)
          .config("spark.sql.iceberg.vectorization.enabled", "false")
          .getOrCreate()
    )
    session.sparkContext.setLogLevel("ERROR")
    print(f"Spark session created | Role: {role} | Auth: {auth_type}")
    return session

---
## 4. DATA_ENGINEER — Full Access Demo (PAT Auth)
Connect as `DATA_ENGINEER` using a Programmatic Access Token. This role has broad access to all schemas and tables.

In [ ]:
spark = create_horizon_spark_session(ENGINEER_PAT, "DATA_ENGINEER")

In [ ]:
print("=== Namespaces visible to DATA_ENGINEER ===")
spark.sql("SHOW NAMESPACES").show(truncate=False)

In [ ]:
print("=== Tables in SALES schema ===")
spark.sql("SHOW TABLES IN horizon.SALES").show(truncate=False)

print("\n=== Tables in ANALYTICS schema ===")
spark.sql("SHOW TABLES IN horizon.ANALYTICS").show(truncate=False)

print("\n=== Tables in RESTRICTED schema ===")
spark.sql("SHOW TABLES IN horizon.RESTRICTED").show(truncate=False)

### 4a. CUSTOMER_ORDERS — Engineer sees ALL regions (no row filtering)

In [ ]:
df_orders = spark.sql("SELECT * FROM horizon.SALES.CUSTOMER_ORDERS ORDER BY order_id")
df_orders.show(truncate=False)
print(f"Total rows: {df_orders.count()} (Engineer sees all regions)")

### 4b. PRODUCT_CATALOG — Full access

In [ ]:
spark.sql("SELECT * FROM horizon.SALES.PRODUCT_CATALOG ORDER BY product_id").show(truncate=False)

### 4c. USER_PROFILES — Masking policies in action
The engineer can access this table, but observe:
- `email`: visible (engineer is allowed)
- `phone`: masked (only ACCOUNTADMIN sees real values)
- `ssn_last4`: always masked as `****`

**Note**: Horizon IRC enforces masking policies on external reads. If your account blocks
external reads on tables with policies, you will see a `NotAuthorized` error instead.

In [ ]:
try:
    spark.sql("SELECT * FROM horizon.ANALYTICS.USER_PROFILES ORDER BY user_id").show(truncate=False)
except Exception as e:
    print(f"Access blocked by policy enforcement: {str(e)[:500]}")
    print("\nThis is expected — Horizon IRC blocks external reads on tables with masking/row policies.")

### 4d. REVENUE_SUMMARY — Restricted financial data (engineer only)

In [ ]:
spark.sql("SELECT * FROM horizon.RESTRICTED.REVENUE_SUMMARY ORDER BY region, quarter").show(truncate=False)

---
## 5. External Write Demo (DATA_ENGINEER)
Write new rows into an Iceberg table from Spark, through Horizon IRC. Requires the
account-level write flags from `01_admin_setup.sql` step 8.

In [ ]:
try:
    spark.sql("""
        INSERT INTO horizon.SALES.PRODUCT_CATALOG VALUES
          (9,  'Ergonomic Chair',  'Furniture',    399.99),
          (10, 'Thunderbolt Dock', 'Accessories',  189.99)
    """)
    print("Write succeeded! New rows inserted via Spark -> Horizon IRC -> Iceberg.")
    spark.sql("SELECT * FROM horizon.SALES.PRODUCT_CATALOG ORDER BY product_id").show(truncate=False)
except Exception as e:
    print(f"Write failed (expected if account write flags not enabled): {str(e)[:500]}")

---
## 6. Key-Pair Authentication Demo
Demonstrates connecting with key-pair auth instead of PAT. This is the production-recommended method.

**Prerequisites** (from `01_admin_setup.sql` step 7):
1. Generate RSA key pair on your machine
2. Assign public key to the Snowflake user
3. Generate a JWT, then exchange for an access token via `/polaris/api/catalog/v1/oauth/tokens`

In [ ]:
# Uncomment and fill in to test key-pair auth:
#
# KEYPAIR_ACCESS_TOKEN = "<access_token_from_jwt_exchange>"
# spark = create_horizon_spark_session(KEYPAIR_ACCESS_TOKEN, "DATA_ENGINEER", auth_type="token")
# spark.sql("SHOW NAMESPACES").show(truncate=False)
# spark.sql("SELECT * FROM horizon.SALES.CUSTOMER_ORDERS LIMIT 5").show()

print("Key-pair auth cell ready. Uncomment and provide access token to test.")

---
## 7. DATA_ANALYST — Restricted Access Demo (PAT Auth)
Connect as `DATA_ANALYST`. This role:
- Can only see the `SALES` schema
- Can read `CUSTOMER_ORDERS` and `PRODUCT_CATALOG`
- **Cannot** access `ANALYTICS` or `RESTRICTED` schemas
- Row access policy filters `CUSTOMER_ORDERS` to only `US-WEST` region

In [ ]:
spark = create_horizon_spark_session(ANALYST_PAT, "DATA_ANALYST")

In [ ]:
print("=== Namespaces visible to DATA_ANALYST ===")
spark.sql("SHOW NAMESPACES").show(truncate=False)

print("\n=== Tables in SALES (analyst's only schema) ===")
spark.sql("SHOW TABLES IN horizon.SALES").show(truncate=False)

### 7a. CUSTOMER_ORDERS — Row access policy in action
The analyst should only see orders where `region = 'US-WEST'`.
Compare with the engineer's view above — same table, fewer rows.

In [ ]:
df_analyst_orders = spark.sql("SELECT * FROM horizon.SALES.CUSTOMER_ORDERS ORDER BY order_id")
df_analyst_orders.show(truncate=False)
print(f"Total rows: {df_analyst_orders.count()} (Analyst sees US-WEST only)")

### 7b. PRODUCT_CATALOG — Analyst has access

In [ ]:
spark.sql("SELECT * FROM horizon.SALES.PRODUCT_CATALOG ORDER BY product_id").show(truncate=False)

### 7c. Unauthorized Access Attempts
The analyst should be denied access to `ANALYTICS.USER_PROFILES` and `RESTRICTED.REVENUE_SUMMARY`.

In [ ]:
print("=== Attempt: ANALYTICS.USER_PROFILES (no grant) ===")
try:
    spark.sql("SELECT * FROM horizon.ANALYTICS.USER_PROFILES LIMIT 5").show()
except Exception as e:
    print(f"BLOCKED (expected): {str(e)[:300]}")

print("\n=== Attempt: RESTRICTED.REVENUE_SUMMARY (no grant) ===")
try:
    spark.sql("SELECT * FROM horizon.RESTRICTED.REVENUE_SUMMARY LIMIT 5").show()
except Exception as e:
    print(f"BLOCKED (expected): {str(e)[:300]}")

---
## 8. Policy Behavior Summary

| Capability | DATA_ENGINEER | DATA_ANALYST |
|---|---|---|
| **Schemas visible** | SALES, ANALYTICS, RESTRICTED | SALES only |
| **CUSTOMER_ORDERS** | All 10 rows (all regions) | US-WEST rows only (row access policy) |
| **PRODUCT_CATALOG** | Full access | Full access |
| **USER_PROFILES** | Access with masking (email visible, phone/SSN masked) | No access (no grant) |
| **REVENUE_SUMMARY** | Full access | No access (no grant) |
| **External Writes** | Allowed (with account flags) | Not granted |

### How Horizon IRC Enforces Snowflake Governance Externally

1. **RBAC**: Spark only sees namespaces/tables the PAT's role has been granted. No data leaks.
2. **Masking Policies**: If the account supports policy-enforced external reads, masked values are returned. Otherwise, access is blocked entirely — fail-secure.
3. **Row Access Policies**: Rows are filtered server-side before Spark receives data. The analyst physically cannot see non-US-WEST rows.
4. **Vended Credentials**: Snowflake generates short-lived cloud storage credentials scoped to the exact data files needed. Spark never gets raw S3 keys.

## 9. Cleanup

In [ ]:
spark.stop()
print("Spark session stopped. Demo complete.")